In [1]:
import os
import shutil
import logging
import matplotlib.pyplot as plt
import numpy as np
dir_path = "../"
os.chdir(dir_path)

from funcs import plot_paths, overlay_paths
from ensemble import Ensemble
from moves import propagate
%matplotlib qt

work_dir = "./longmd_rugged_br50M/"

if os.path.exists(work_dir):
    shutil.rmtree(work_dir)
if not os.path.exists(work_dir):
    os.mkdir(work_dir)
# copy all py files of cwd to test
for file in os.listdir(os.getcwd()):
    if file.endswith(".py"):
        shutil.copy(file, work_dir)
os.chdir(work_dir)
print(os.getcwd())

/mnt/0bf0c339-34bb-4500-a5fb-f3c2a863de29/DATA/PyRETIS3/toytis/longmd_rugged_br50M


In [2]:
# intfs = [-1., -0.75, -0.5, -0.25, 0., 0.25, 0.5, 0.75, 1., 1.25]
intfs = [-0.1, 0., 0.1, 0.2, 0.3, 0.4, 0.5]
# intfs = [-1., -0.75, -0.5, -0.25, 0.]

# Set-up the simulation keys for RETIS
len = 50000000
mdset = {"intfs" : {"L": 0, "R":0, "M":0},
            "ens_type" : "RETIS_0plus", 
            "simtype" : "retis",
            "id" : 1,
            "method" : "load",
            "max_len" : 100000,
            "dt": 0.0002,
            "temperature": 1.,
            "dim" : 1,
            "mass" : 1.,
            "friction": 5.,
            "high_friction": True,
            "max_cycles": 10000000,
            "p_shoot": 0.8,
            "include_stateB": False,
            'prime_both_starts': False,
            'snake_Lmax': 0,
            'max_paths': 5,
            'save_pe2': False,
            'pe2_N': 0
}

logger = logging.getLogger()
file_handler = logging.FileHandler("logging.log")
formatter = logging.Formatter('[%(levelname)s] %(name)s %(funcName)s %(lineno)d: %(message)s')
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.setLevel(logging.INFO)
logger.info("Hello world!")
logger.info("\ninterfaces = {}\n".format(intfs) +
            #  "zero_left = {}\n".format(istarset["zero_left"]) +
             "timestep = {}\n".format(mdset["dt"]))


dummy_ens = Ensemble(mdset)
dummy_ens.end_conditions = {}; dummy_ens.start_conditions = {}; dummy_ens.cross_conditions = {}; dummy_ens.extremal_conditions = {}


start = (-0.105, dummy_ens.engine.draw_velocities())
ord_seeve = []
len_i = 0

while len - len_i > 0:
    len -= len_i
    len_i = min(25000000, len) 
    _, md_tuple = propagate(dummy_ens, start, 1, len_i)
    ord_seeve += md_tuple[1][::1]
    start = md_tuple[0][-1]
    md_tuple = []

np.save("md_path.npy", ord_seeve)

# set loglevel to warning for matplotlib
logging.getLogger("matplotlib").setLevel(logging.WARNING)

# Check the simulation output in the pathensemble.txt files 
# The output is compatible with pyretisanalyse. But you'll have to change
# the dirnames from [0, 1, 2, ...] --> [000, 001, 002, ...]

In [ ]:
# plot the potential and the interfaces
fig,ax =plt.subplots()
dummy_ens.engine.potential.plot_potential(ax)
for intf in intfs:
    ax.axvline(intf, color="black", linestyle="--")
fig.show()

AttributeError: 'FlatWall1D' object has no attribute 'plot_potential'

In [ ]:
# A Langevin engine is used. This can be swapped for a velocity verlet engine if 
# you want NVE dynamics. (But you'll have to change this in ensemble.py, where
# the engine object is initialized.)